# YZTA Datathon — v4 Optimized

> **Tüm pipeline tek akışta.** Hücreleri **sırayla** çalıştır. Kernel restart sonrası en baştan başla.

## HÜCRE 1 — Kurulum & İmportlar

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm', 'catboost', 'optuna', '-q'],
               capture_output=True)

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('Tum kutuphaneler yuklendi.')

## HÜCRE 2 — Veri Yükleme

In [ ]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test_x.csv')

train_id = train_raw['id'].copy()
test_id  = test_raw['id'].copy()
target   = train_raw['bilissel_performans_skoru'].copy()

train = train_raw.drop(columns=['id', 'bilissel_performans_skoru']).copy()
test  = test_raw.drop(columns=['id']).copy()

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Target Min:{target.min():.2f} Max:{target.max():.2f} Ort:{target.mean():.2f} Std:{target.std():.2f}')
print(f'Skewness: {target.skew():.4f}')

## HÜCRE 3 — Log Transform Kararı

In [ ]:
skewness   = target.skew()
target_min = target.min()
SHIFT      = max(0, -target_min + 0.01)
USE_LOG    = abs(skewness) > 0.75

print(f'Skewness: {skewness:.4f} -- Log Transform: {"ACIK" if USE_LOG else "KAPALI"}')
if USE_LOG:
    target_transformed = np.log1p(target + SHIFT)
    print(f'Shift: {SHIFT:.4f} | Log target Min:{target_transformed.min():.3f} Max:{target_transformed.max():.3f}')
else:
    target_transformed = target.copy()
    SHIFT = 0
    print('Log transform uygulanmadi.')

## HÜCRE 4 — Temizlik & Encoding

In [ ]:
ulke_mapping   = {'spain':'ispanya','south korea':'guney kore','sweden':'isvec',
                  'netherlands':'hollanda','mexico':'meksika','china':'cin'}
meslek_mapping = {'lawyer':'avukat'}

cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(include=['int64','float64']).columns.tolist()

for col in cat_cols:
    train[col] = train[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()
    test[col]  = test[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()

train['ulke']   = train['ulke'].replace(ulke_mapping)
test['ulke']    = test['ulke'].replace(ulke_mapping)
train['meslek'] = train['meslek'].replace(meslek_mapping)
test['meslek']  = test['meslek'].replace(meslek_mapping)

for col in num_cols:
    med        = train[col].median()
    train[col] = train[col].fillna(med)
    test[col]  = test[col].fillna(med)

for col in num_cols:
    lo, hi     = train[col].quantile(0.01), train[col].quantile(0.99)
    train[col] = train[col].clip(lo, hi)
    test[col]  = test[col].clip(lo, hi)

# Binary encoding
for col in ['cinsiyet', 'gun_tipi']:
    cats       = sorted(pd.concat([train[col], test[col]], ignore_index=True).unique())
    mp         = {c: i for i, c in enumerate(cats)}
    train[col] = train[col].map(mp)
    test[col]  = test[col].map(mp)
    print(f'  {col}: {mp}')

# OOF Smoothed Target Encoding
TARGET_ENCODE_COLS = ['kronotip', 'ruh_sagligi_durumu', 'meslek', 'ulke', 'mevsim']
SMOOTH             = 15
global_mean        = target_transformed.mean()
kf_enc             = KFold(n_splits=5, shuffle=True, random_state=SEED)

for col in TARGET_ENCODE_COLS:
    agg    = pd.DataFrame({'col': train[col].values, 'target': target_transformed.values})
    stats  = agg.groupby('col')['target'].agg(['mean', 'count'])
    sm_map = ((stats['count'] * stats['mean'] + SMOOTH * global_mean)
              / (stats['count'] + SMOOTH)).to_dict()

    oof_enc = np.full(len(train), global_mean, dtype=np.float64)
    for tr_i, va_i in kf_enc.split(train):
        fa       = agg.iloc[tr_i]
        fs       = fa.groupby('col')['target'].agg(['mean', 'count'])
        fm       = ((fs['count'] * fs['mean'] + SMOOTH * global_mean)
                    / (fs['count'] + SMOOTH)).to_dict()
        oof_enc[va_i] = train[col].iloc[va_i].map(fm).fillna(global_mean).values

    train[col] = oof_enc
    test[col]  = test[col].map(sm_map).fillna(global_mean)
    print(f'  {col}: target encoding tamam')

print(f'\nTemizlik & encoding bitti. Train:{train.shape} | Test:{test.shape}')

## HÜCRE 5 — Feature Engineering (8 mevcut + 5 yeni)

In [ ]:
def add_features(df):
    df = df.copy()
    # -- Mevcut v3 --
    df['meslek_gun_tipi']      = df['meslek'] * df['gun_tipi']
    df['uyku_kalite_endeksi']  = (df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']) / (df['gecelik_uyanma_sayisi'] + 1)
    df['toplam_kaliteli_uyku'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['zihinsel_yuk']         = df['stres_skoru'] * df['gunluk_calisma_saati']
    df['uyku_bozulma_skoru']   = df['gecelik_uyanma_sayisi'] * df['uykuya_dalma_suresi_dk']
    df['ekran_kafein']         = df['uyku_oncesi_ekran_suresi_dk'] * df['uyku_oncesi_kafein_mg']
    df['stres_uyku_orani']     = df['stres_skoru'] / (df['uyku_kalite_endeksi'] + 1)
    df['yas_stres']            = df['yas'] * df['stres_skoru']
    # -- Yeni v4 --
    df['stres_aktivite_dengesi'] = df['gunluk_adim_sayisi'] / (df['stres_skoru'] + 1)
    df['uyku_kalite_kuvvet']     = df['uyku_kalite_endeksi'] * df['toplam_kaliteli_uyku']
    df['bmi_aktivite']           = df['vucut_kitle_indeksi'] / (df['gunluk_adim_sayisi'] / 1000 + 1)
    df['stres_uyku_gecikme']     = df['stres_skoru'] * df['uykuya_dalma_suresi_dk']
    df['kafein_uyanma_birikimi'] = df['uyku_oncesi_kafein_mg'] * (df['gecelik_uyanma_sayisi'] + 1)
    return df

train = add_features(train)
test  = add_features(test)
print(f'Feature engineering tamam. Toplam ozellik: {train.shape[1]}')

## HÜCRE 6 — Feature Selection

In [ ]:
sel = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                         num_leaves=31, random_state=SEED, verbose=-1)
sel.fit(train, target_transformed)

imp_df = (pd.DataFrame({'ozellik': train.columns,
                         'importance': sel.feature_importances_})
            .sort_values('importance', ascending=False))
print('Feature Importance (ilk 20):')
print(imp_df.head(20).to_string(index=False))

drop_cols = imp_df[imp_df['importance'] == 0]['ozellik'].tolist()
print(f'\nAtilacak: {drop_cols}')
train = train.drop(columns=drop_cols, errors='ignore')
test  = test.drop(columns=drop_cols, errors='ignore')
print(f'Kalan ozellik: {train.shape[1]}')

## HÜCRE 7 — Optuna Hyperparameter Tuning

In [ ]:
X       = train.copy()
y       = target_transformed.copy()
kf_tune = KFold(n_splits=5, shuffle=True, random_state=SEED)

def lgb_objective(trial):
    params = {
        'objective'        : 'regression',
        'metric'           : 'rmse',
        'verbosity'        : -1,
        'random_state'     : SEED,
        'bagging_freq'     : 5,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 80),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.0, 2.0),
    }
    oof = np.zeros(len(X))
    for tr_i, va_i in kf_tune.split(X):
        dt = lgb.Dataset(X.iloc[tr_i], label=y.iloc[tr_i])
        dv = lgb.Dataset(X.iloc[va_i], label=y.iloc[va_i], reference=dt)
        m  = lgb.train(params, dt, num_boost_round=2000, valid_sets=[dv],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(-1)])
        oof[va_i] = m.predict(X.iloc[va_i])
    return np.sqrt(mean_squared_error(y, oof))

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lgb_objective, n_trials=50)

best_lgb_params = study.best_params
best_lgb_params.update({'objective':'regression', 'metric':'rmse',
                         'verbosity':-1, 'random_state':SEED, 'bagging_freq':5})
print(f'En iyi LGB CV RMSE : {study.best_value:.5f}')
print(f'Parametreler       : {best_lgb_params}')

## HÜCRE 8 — 5-Fold CV Ensemble Eğitimi

In [ ]:
N_FOLDS  = 5
kf_model = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

lgb_oof  = np.zeros(len(X));   xgb_oof  = np.zeros(len(X));   cat_oof  = np.zeros(len(X))
lgb_test = np.zeros(len(test)); xgb_test = np.zeros(len(test)); cat_test = np.zeros(len(test))
lgb_sc, xgb_sc, cat_sc = [], [], []

xgb_params = dict(
    n_estimators=3000,
    learning_rate=best_lgb_params.get('learning_rate', 0.01),
    max_depth=5,
    min_child_weight=10,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=best_lgb_params.get('feature_fraction', 0.75),
    colsample_bylevel=0.8,
    reg_alpha=0.2,
    reg_lambda=1.5,
    early_stopping_rounds=150,
    eval_metric='rmse',
    random_state=SEED,
    verbosity=0,
)

print('=== 5-Fold Ensemble Egitimi ===\n')
for fold, (tr_i, va_i) in enumerate(kf_model.split(X)):
    Xtr, Xva = X.iloc[tr_i], X.iloc[va_i]
    ytr, yva = y.iloc[tr_i], y.iloc[va_i]
    print(f'--- Fold {fold+1} ---')

    # LightGBM
    dt = lgb.Dataset(Xtr, label=ytr)
    dv = lgb.Dataset(Xva, label=yva, reference=dt)
    lm = lgb.train(best_lgb_params, dt, num_boost_round=3000, valid_sets=[dv],
                   callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)])
    lgb_oof[va_i]  = lm.predict(Xva)
    lgb_test      += lm.predict(test) / N_FOLDS
    s = np.sqrt(mean_squared_error(yva, lgb_oof[va_i]))
    lgb_sc.append(s); print(f'  LGB  RMSE: {s:.5f}')

    # XGBoost
    xm = XGBRegressor(**xgb_params)
    xm.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    xgb_oof[va_i]  = xm.predict(Xva)
    xgb_test      += xm.predict(test) / N_FOLDS
    s = np.sqrt(mean_squared_error(yva, xgb_oof[va_i]))
    xgb_sc.append(s); print(f'  XGB  RMSE: {s:.5f}')

    # CatBoost
    cm = CatBoostRegressor(
        iterations=3000,
        learning_rate=best_lgb_params.get('learning_rate', 0.01),
        depth=6,
        min_data_in_leaf=best_lgb_params.get('min_child_samples', 40),
        subsample=best_lgb_params.get('bagging_fraction', 0.75),
        l2_leaf_reg=3.0,
        early_stopping_rounds=150,
        eval_metric='RMSE',
        random_seed=SEED,
        verbose=False,
    )
    cm.fit(Xtr, ytr, eval_set=(Xva, yva))
    cat_oof[va_i]  = cm.predict(Xva)
    cat_test      += cm.predict(test) / N_FOLDS
    s = np.sqrt(mean_squared_error(yva, cat_oof[va_i]))
    cat_sc.append(s); print(f'  CAT  RMSE: {s:.5f}\n')

lgb_r = np.sqrt(mean_squared_error(y, lgb_oof))
xgb_r = np.sqrt(mean_squared_error(y, xgb_oof))
cat_r = np.sqrt(mean_squared_error(y, cat_oof))

print('=== CV Ozeti ===')
print(f'LGB  OOF RMSE: {lgb_r:.5f}  (+-{np.std(lgb_sc):.5f})')
print(f'XGB  OOF RMSE: {xgb_r:.5f}  (+-{np.std(xgb_sc):.5f})')
print(f'CAT  OOF RMSE: {cat_r:.5f}  (+-{np.std(cat_sc):.5f})')

## HÜCRE 9 — Ensemble: Weighted Avg vs Ridge (oto-seçim)

In [ ]:
# Weighted Averaging
rmses   = np.array([lgb_r, xgb_r, cat_r])
inv_r   = 1.0 / rmses
wa_w    = inv_r / inv_r.sum()
wa_oof  = wa_w[0]*lgb_oof  + wa_w[1]*xgb_oof  + wa_w[2]*cat_oof
wa_test = wa_w[0]*lgb_test + wa_w[1]*xgb_test + wa_w[2]*cat_test
wa_rmse = np.sqrt(mean_squared_error(y, wa_oof))
print(f'Weighted Avg  agirliklar  LGB:{wa_w[0]:.3f}  XGB:{wa_w[1]:.3f}  CAT:{wa_w[2]:.3f}')
print(f'Weighted Avg  OOF RMSE  : {wa_rmse:.5f}')

# Ridge Stacking
meta_tr    = np.column_stack([lgb_oof,  xgb_oof,  cat_oof])
meta_te    = np.column_stack([lgb_test, xgb_test, cat_test])
ridge      = Ridge(alpha=1.0, fit_intercept=True)
ridge.fit(meta_tr, y)
ridge_oof  = ridge.predict(meta_tr)
ridge_rmse = np.sqrt(mean_squared_error(y, ridge_oof))
print(f'\nRidge Stacking OOF RMSE : {ridge_rmse:.5f}')
print(f'Ridge katsayilar  LGB:{ridge.coef_[0]:.4f}  XGB:{ridge.coef_[1]:.4f}  CAT:{ridge.coef_[2]:.4f}')

# Otomatik secim
if wa_rmse <= ridge_rmse:
    final_test_log = wa_test
    best_method    = f'Weighted Averaging  RMSE={wa_rmse:.5f}'
else:
    final_test_log = ridge.predict(meta_te)
    best_method    = f'Ridge Stacking  RMSE={ridge_rmse:.5f}'

print(f'\nSECILEN STRATEJI: {best_method}')

## HÜCRE 10 — Submission

In [ ]:
if USE_LOG:
    test_preds = np.expm1(final_test_log) - SHIFT
else:
    test_preds = final_test_log

test_preds = np.clip(test_preds, target.min(), target.max())

print(f'Min:{test_preds.min():.4f}  Max:{test_preds.max():.4f}  Ort:{test_preds.mean():.4f}  Std:{test_preds.std():.4f}')

submission = pd.DataFrame({'id': test_id, 'bilissel_performans_skoru': test_preds})
submission.to_csv('submission.csv', index=False)
print(f'submission.csv kaydedildi -- {len(submission)} satir')
print(submission.head())